# IDX Exchange DA Internship - Week Three - Mortgage Rate Enrichment

Submit a .py script that: 
* Fetches the MORTGAGE30US series directly from FRED
* Resamples it to monthly averages
* Merges it onto both the combined sold and listings datasets using a year_month key
* Includes a validation check confirming no null rate values exist after the merge

Save both enriched datasets as new CSVs

In [26]:
import pandas as pd

## Import Datasets

In [27]:
# MLS Datasets

listings = pd.read_csv("week_2_listings.csv", low_memory=False)
sold = pd.read_csv("week_2_sold.csv", low_memory=False)

In [28]:
# Fetch FRED Dataset

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=["observation_date"])
mortgage.columns = ["date", "rate_30yr_fixed"]

mortgage

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29
...,...,...
2893,2026-09-10,6.76
2894,2026-09-17,6.95
2895,2026-09-24,7.03
2896,2026-10-01,7.28


## Restructure FRED Dataset

In [29]:
# FRED database is updated on a weekly basis. MLS Dataset is based on monthly data
# Restructure FRED database from weekly to monthly averages

mortgage["year_month"] = mortgage["date"].dt.to_period("M")
mortgage_monthly = (mortgage.groupby("year_month")["rate_30yr_fixed"].mean().reset_index())

mortgage_monthly

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975
...,...,...
662,2026-06,6.4900
663,2026-07,6.5420
664,2026-08,6.6675
665,2026-09,6.8625


## Create Matching Keys in the MLS Data

In [30]:
# For listing data, create a key from ListingContractDate

listings["year_month"] = pd.to_datetime(listings["ListingContractDate"]).dt.to_period("M")

In [31]:
# For sold data, create a key from CloseDate

sold["year_month"] = pd.to_datetime(sold["CloseDate"]).dt.to_period("M")

## Merge FRED and MLS

In [32]:
# Use the newly created key to add the FRED data to both the listings and sold datasets

sold_with_rates = sold.merge(mortgage_monthly, on="year_month", how="left")
listings_with_rates = listings.merge(mortgage_monthly, on="year_month", how="left")

## Validate the Merge

In [33]:
# Same amount of rows and one more column is a good sign that the keys aligned correctly

print(listings.shape, listings_with_rates.shape)
print(sold.shape, sold_with_rates.shape)

(547162, 71) (547162, 72)
(414054, 74) (414054, 75)


In [34]:
# No null values is also a good sign that the keys all aligned

print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


In [35]:
listings_with_rates[["ListingContractDate", "year_month", "rate_30yr_fixed"]].head()

,ListingContractDate,year_month,rate_30yr_fixed
0,2024-01-01,2024-01,6.6425
1,2024-01-24,2024-01,6.6425
2,2024-01-12,2024-01,6.6425
3,2024-01-20,2024-01,6.6425
4,2024-01-12,2024-01,6.6425


In [36]:
sold_with_rates[["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]].head()

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-26,2024-01,240000.0,6.6425
1,2024-01-05,2024-01,815000.0,6.6425
2,2024-01-05,2024-01,810000.0,6.6425
3,2024-01-02,2024-01,2100000.0,6.6425
4,2024-01-22,2024-01,1950000.0,6.6425


In [37]:
# Save new datasets

listings_with_rates.to_csv("listings_with_rates.csv", index=False)
sold_with_rates.to_csv("sold_with_rates.csv", index=False)